<a href="https://colab.research.google.com/github/fariha34/BRACU-CSE/blob/main/BRACU-CSE330/Linear_Systems_lab7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1: Solving a linear system using inverse matrix

We have a linear system

\begin{align}
&a_{11} x_1 + a_{12} x_2 +  \cdots  + a_{1n} x_n = b_1\\
&a_{21} x_1 + a_{22} x_2 +  \cdots + a_{2n} x_n = b_2\\
&\cdots\\
&a_{n1} x_1 + a_{n2} x_2 +  \cdots + a_{nn} x_n = b_n\\
\end{align}

It is convenient to express this system in the matrix form

\begin{align}
Ax = b
\end{align}

where $A$ is an $n \times n$ square matrix with elements $a_{ij}$, and $x$, $b$ are $n \times 1$ vectors.

This system has a unique solution iff $A$ is non-singular, given by $x = A^{-1}b.$

In [1]:
import numpy as np

In [2]:
def get_result_by_inverse_matrix(A, b):
    # A and b are numpy arrays
    # first check if A is non-singular: its determinant must be non-zero
    d = np.linalg.det(A)   # find determinant
    if np.isclose(d, 0):
        raise ValueError("Singular Matrix, inverse not possible!")
    A_inv = np.linalg.inv(A)
    x = np.dot(A_inv, b)
    return x

In [3]:
# Test case for get_result_by_inverse_matrix(A, b)
data_A = np.array([[1, 2, 1], [1, -2, 2], [2, 12, -2]])
data_b = np.array([0, 4, 4])

test = get_result_by_inverse_matrix(data_A, data_b)
print(test)
np.testing.assert_allclose(test, [11, -2.5, -6])

[11.  -2.5 -6. ]


# Part 2: Gaussian elimination

Gaussian elimination uses elementary row operations to transform the system to upper triangular form $Ux = y$.

Elementary row operations include swapping rows and adding multiples of one row to another. They don't change the solution $x$, but they change the matrix $A$ and the right-hand side $b$.

The upper triangular matrix $U$ is

\begin{bmatrix}
u_{11} & u_{12} & \cdots & u_{1n}\\
0 & u_{22} & \cdots & u_{2n}\\
\vdots & \vdots & \ddots & \vdots\\
0 & \cdots & 0 & u_{nn}\\
\end{bmatrix}

**Algorithm**

Let $A^{(1)}=A$ and $b^{(1)}=b$. For each $k$ from 1 to $n-1$:

1. Define the row multipliers
$$m_{ik} = \frac{a_{ik}^{(k)}}{a_{kk}^{(k)}}, \quad i=k+1,\dots,n.$$
2. Use them to remove $x_k$ from equations $k+1$ to $n$:
$$a_{ij}^{(k+1)}=a_{ij}^{(k)}-m_{ik}a_{kj}^{(k)}, \quad b_{i}^{(k+1)}=b_{i}^{(k)}-m_{ik}b_{k}^{(k)}, \quad i,j=k+1,\dots, n.$$

The row operations are applied to the augmented matrix $[A \mid b]$, which handles $A$ and $b$ together.

Then **backward substitution** solves the upper triangular system $Ux = b$:

$$x_j = \frac{b_j-\sum_{k=j+1}^{n}u_{jk}x_k}{u_{jj}}, \quad j = n, n-1, \dots, 1.$$

In [4]:
def get_result_gaussian_elimination(n, A):
    # n is the number of unknowns
    # A is the n x (n+1) augmented matrix
    A = np.array(A, dtype=float)   # float copy, so integer input isn't truncated
    x = np.zeros(n)

    # Gaussian elimination: make A upper triangular (i = column, j = row)
    for i in range(n):
        if A[i, i] == 0:
            raise ValueError("Zero pivot: calculation is not possible!")
        for j in range(i + 1, n):
            m = A[j, i] / A[i, i]
            A[j] = A[j] - m * A[i]

    # Back substitution (j = row, k = column)
    for j in range(n - 1, -1, -1):
        s = 0
        for k in range(j + 1, n):
            s += A[j][k] * x[k]
        x[j] = (A[j][n] - s) / A[j][j]

    return x

In [5]:
# Test case for get_result_gaussian_elimination(n, A)
data_n = 3
data_A = np.array([[1, 2, 1, 0], [1, -2, 2, 4], [2, 12, -2, 4]])

test = get_result_gaussian_elimination(data_n, data_A)
print(test)
np.testing.assert_allclose(test, [11, -2.5, -6])

[11.  -2.5 -6. ]


# Part 3: LU Decomposition

Transform the $n \times n$ matrix $A$ into a product of a lower triangular matrix $L$ and an upper triangular matrix $U$:

$$A = LU$$

**Algorithm**

1. Initialize $L$ to the $n \times n$ identity matrix and $U = A$.
2. For $i = 1, \dots, n$:
3. &nbsp;&nbsp; For $j = i+1, \dots, n$:
4. &nbsp;&nbsp;&nbsp;&nbsp; Set $l_{ji} = u_{ji}/u_{ii}$
5. &nbsp;&nbsp;&nbsp;&nbsp; Set $U_j = U_j - l_{ji}U_i$ (rows $i$ and $j$ of $U$)

Then $Ax = b$ becomes $LUx = b$. Let $Ux = y$:

* solve $Ly = b$ for $y$ by **forward substitution** ($L$ is lower triangular)
* solve $Ux = y$ for $x$ by **backward substitution** ($U$ is upper triangular)

In [6]:
def lu(A):
    n = A.shape[0]
    U = A.astype(float).copy()
    L = np.eye(n, dtype=np.double)

    for i in range(n):
        for j in range(i + 1, n):
            L[j, i] = U[j, i] / U[i, i]
            U[j] = U[j] - L[j, i] * U[i]
    return L, U


def forward_substitution(L, b):
    n = L.shape[0]
    y = np.zeros_like(b, dtype=np.double)
    y[0] = b[0] / L[0, 0]
    for i in range(1, n):
        y[i] = (b[i] - L[i, :i] @ y[:i]) / L[i, i]
    return y


def back_substitution(U, y):
    n = U.shape[0]
    x = np.zeros_like(y, dtype=np.double)
    x[-1] = y[-1] / U[-1, -1]
    for i in range(n - 2, -1, -1):
        x[i] = (y[i] - U[i, i + 1:] @ x[i + 1:]) / U[i, i]
    return x


def lu_solve(A, b):
    L, U = lu(A)
    y = forward_substitution(L, b)
    return back_substitution(U, y)

In [7]:
# Test case for lu_solve(A, b)
data_A = np.array([[1., 2., 1.], [1., -2., 2.], [2., 12., -2.]])
data_b = np.array([0., 4., 4.])

L, U = lu(data_A)
print("L =\n", L)
print("U =\n", U)
np.testing.assert_allclose(L @ U, data_A)

test = lu_solve(data_A, data_b)
print("x =", test)
np.testing.assert_allclose(test, [11, -2.5, -6])

L =
 [[ 1.  0.  0.]
 [ 1.  1.  0.]
 [ 2. -2.  1.]]
U =
 [[ 1.  2.  1.]
 [ 0. -4.  1.]
 [ 0.  0. -2.]]
x = [11.  -2.5 -6. ]


# Comparing all three methods

In [8]:
print("Inverse matrix      :", get_result_by_inverse_matrix(data_A, data_b))
print("Gaussian elimination:", get_result_gaussian_elimination(3, np.column_stack([data_A, data_b])))
print("LU decomposition    :", lu_solve(data_A, data_b))
print("numpy.linalg.solve  :", np.linalg.solve(data_A, data_b))

Inverse matrix      : [11.  -2.5 -6. ]
Gaussian elimination: [11.  -2.5 -6. ]
LU decomposition    : [11.  -2.5 -6. ]
numpy.linalg.solve  : [11.  -2.5 -6. ]
